# Comparer des agents (local / Colab, sans submit)

Sur **Colab**, le runtime distant n'a pas ton `.venv` ni `src/` : exécute d'abord la cellule bootstrap.

In [ ]:
# Bootstrap : local → src/ ; Colab → clone GitHub + sys.path
import runpy
import sys
from pathlib import Path

_bootstrap = Path("colab_bootstrap.py")
if not _bootstrap.exists():
    _bootstrap = Path("notebooks/colab_bootstrap.py")

if _bootstrap.exists():
    runpy.run_path(str(_bootstrap), run_name="__main__")
else:
    import getpass
    import os
    import subprocess

    REPO, ROOT = "matleniz/python_deepL", Path("/content/python_deepL")
    src = None
    for p in [
        Path.cwd() / "src",
        Path.cwd().parent / "src",
        Path.home() / "python_deepL" / "src",
        ROOT / "src",
    ]:
        if (p / "projet").is_dir():
            src = p.resolve()
            break
    if src is None:
        tok = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub PAT (repo) : ")
        if not ROOT.exists():
            subprocess.check_call([
                "git", "clone", "--depth", "1",
                f"https://{tok}@github.com/{REPO}.git", str(ROOT),
            ])
        else:
            subprocess.check_call(["git", "-C", str(ROOT), "pull", "--ff-only"])
        src = ROOT / "src"
    sys.path.insert(0, str(src))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "pettingzoo", "pygame", "numpy",
    ])
    print("projet OK ←", src)

In [ ]:
from projet.play import compare, duel, evaluate
from projet.agents.random_agent import Agent as RandomAgent
from projet.agents.monte_carlo import Agent as MCAgent

# from projet.agents.nn_pytorch import Agent as NNAgent
# from projet.agents.rl_agent import Agent as RLAgent

N = 40
mc = lambda: MCAgent(n_sim=30)

## vs random

In [ ]:
for name, factory in [("random", RandomAgent), ("MCplat", mc)]:
    mean, wr = evaluate(factory, n=N)
    print(f"{name:8s}  vs random  mean={mean:+.3f}  winrate={wr:.3f}  (n={N})")

## Face à face

In [ ]:
stats = compare(mc, RandomAgent, n=N)
print("MCplat (A) vs random (B)")
for k, v in stats.items():
    print(f"  {k}: {v}")

## Une partie

In [ ]:
r = duel(mc, RandomAgent, seed=0, a_seat=0)
print(f"seed=0, A=MCplat seat0 → reward_A={r:+.0f}")

## Matrice

In [ ]:
agents = {
    "random": RandomAgent,
    "MCplat": mc,
}

names = list(agents)
print("winrate A\\B".ljust(10), *[f"{n:>8s}" for n in names])
for na in names:
    row = []
    for nb in names:
        if na == nb:
            row.append("   —   ")
        else:
            s = compare(agents[na], agents[nb], n=N)
            row.append(f"{s['winrate_a']:8.3f}")
    print(f"{na:10s}", *row)